# Swiss Legal Citation Retrieval — Dense Embedding Maximum-Context Ceiling Test
## Single SOTA model, full corpus, every signal from `data_insights/` injected

**The question this notebook answers, definitively:**
> Given the strongest open multilingual encoder available in 2026, the full ~2.6M Swiss legal
> corpus, AND every structural/contextual signal we have pre-extracted in `data_insights/`,
> can dense retrieval alone reach the Macro F1 target of 0.60–0.80 on the val set?

**Hardware:** 95 GB VRAM single GPU. fp16 throughout, no quantization.

**Model:** `Qwen/Qwen3-Embedding-8B` — #1 on MMTEB Multilingual leaderboard (June 2025, 70.58),
beats Gemini-Embedding-001 by 2+ points on multilingual mean-task. Apache 2.0.

**Why a single model:** This is hypothesis testing, not benchmarking. If the SOTA encoder with
maximum context cannot clear the target, no weaker model can either, and Observation 3 is
confirmed with the strongest possible evidence.

---

## Why this notebook is different from a vanilla dense retrieval test

Each corpus passage is enriched with three additional signals from `data_insights/` before encoding,
so the encoder sees the **full legal context** of each citation, not just its terse statute text:

| Signal | Source file | What it adds |
|---|---|---|
| **Structured citation header** | `*_classified_citations.jsonl` | Parsed segments: `law_code`, `article_number`, `paragraph`, `subdivision`, `decision_year`, `legal_area_code` — gives the encoder explicit semantic anchors |
| **Outgoing reference list** | `*_links.json` | The other citations that THIS citation references in its body — captures the legal neighborhood |
| **Original title + text** | `data/laws_de.csv` (`title` column) | Often missing from raw text but contains the law's domain (e.g., "Bundesgesetz über den Umweltschutz") |

Each enriched passage looks like:
```
[CITATION: Art. 11 Abs. 2 OR — Bundesgesetz betreffend die Ergänzung des Schweizerischen Zivilgesetzbuches]
[STRUCTURE: Article 11, paragraph 2, law code OR (Obligationenrecht), pattern statute_article]
[REFERENCES: Art. 1 OR; Art. 12 OR; Art. 24 OR]
[TEXT: ...original German text of the article...]
```

This is the maximum amount of context we can inject without changing the retrieval architecture.
If dense embedding cannot reach the target with this much help, it definitively cannot reach it
with less.

---

## Required input files (upload to Drive)

| File | Size | From |
|---|---|---|
| `val.csv` | small | `data/` |
| `laws_de.csv` | ~50 MB | `data/` |
| `court_considerations.csv` | ~2 GB | `data/` |
| `gold_citation_coverage.csv` | 316 KB | `data_insights/` |
| `laws_de_classified_citations.jsonl` | 80 MB | `data_insights/` |
| `court_considerations_classified_citations.jsonl` | 817 MB | `data_insights/` |
| `laws_de_links.json` | 2.6 MB | `data_insights/` |
| `court_considerations_links.json` | 142 MB | `data_insights/` |

All five `data_insights` files are essential — each adds a different kind of context to the passages.

---

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 0. Environment

In [2]:
!nvidia-smi

Sun Apr 26 12:11:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
!pip uninstall -y numpy
!pip install -q -U numpy==1.26.4 sentence-transformers transformers accelerate faiss-cpu pandas ijson
# flash-attention-2 for 2-3x speedup on long sequences (H100/A100/Blackwell)
!pip install -q flash-attn --no-build-isolation || echo "flash-attn install failed; will fall back to sdpa"

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 131.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which

In [4]:
import os, gc, time, json, math, warnings
import numpy as np
import pandas as pd
import torch
warnings.filterwarnings('ignore')

DEVICE = 'cuda'
assert torch.cuda.is_available(), 'GPU required'
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

GPU : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 102.0 GB


In [5]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR     = '/content/drive/MyDrive/swiss_law/data'
INSIGHTS_DIR = '/content/drive/MyDrive/swiss_law/data_insights'
ART_DIR      = '/content/drive/MyDrive/swiss_law/artifacts'
os.makedirs(ART_DIR, exist_ok=True)

FILES = {
    'val'                     : f'{DATA_DIR}/val.csv',
    'laws'                    : f'{DATA_DIR}/laws_de.csv',
    'court'                   : f'{DATA_DIR}/court_considerations.csv',
    'coverage'                : f'{INSIGHTS_DIR}/gold_citation_coverage.csv',
    'laws_classified'         : f'{INSIGHTS_DIR}/laws_de_classified_citations.jsonl',
    'court_classified'        : f'{INSIGHTS_DIR}/court_considerations_classified_citations.jsonl',
    'laws_links'              : f'{INSIGHTS_DIR}/laws_de_links.json',
    'court_links'             : f'{INSIGHTS_DIR}/court_considerations_links.json',
}
for tag, p in FILES.items():
    print(f'  {"OK" if os.path.exists(p) else "MISSING":>7}  {tag:25s} {p}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
       OK  val                       /content/drive/MyDrive/swiss_law/data/val.csv
       OK  laws                      /content/drive/MyDrive/swiss_law/data/laws_de.csv
       OK  court                     /content/drive/MyDrive/swiss_law/data/court_considerations.csv
       OK  coverage                  /content/drive/MyDrive/swiss_law/data_insights/gold_citation_coverage.csv
       OK  laws_classified           /content/drive/MyDrive/swiss_law/data_insights/laws_de_classified_citations.jsonl
       OK  court_classified          /content/drive/MyDrive/swiss_law/data_insights/court_considerations_classified_citations.jsonl
       OK  laws_links                /content/drive/MyDrive/swiss_law/data_insights/laws_de_links.json
       OK  court_links               /content/drive/MyDrive/swiss_law/data_insights/court_considerations_links.json


## 1. Load base data (val + corpus CSVs)

In [6]:
print('Loading val.csv ...')
val = pd.read_csv(FILES['val'])

print('Loading laws_de.csv ...')
laws = pd.read_csv(FILES['laws'])
laws['source'] = 'laws'

print('Loading court_considerations.csv (~2.4M rows) ...')
court = pd.read_csv(FILES['court'])
court['source'] = 'court'
court['title']  = ''

laws  = laws[['citation','text','title','source']].fillna('')
court = court[['citation','text','title','source']].fillna('')
corpus = (
    pd.concat([laws, court], ignore_index=True)
      .drop_duplicates(subset='citation')
      .reset_index(drop=True)
)
print(f'\nCorpus size: {len(corpus):,}')
print(corpus['source'].value_counts())
cit_to_idx = {c: i for i, c in enumerate(corpus['citation'].values)}

Loading val.csv ...
Loading laws_de.csv ...
Loading court_considerations.csv (~2.4M rows) ...

Corpus size: 2,161,111
source
court    1985178
laws      175933
Name: count, dtype: int64


## 2. Load `data_insights/` — classified citations and link graphs

These are merged into the corpus to enrich each passage.

In [7]:
# --- 2a. Classified citation segments (one dict per citation) ---
def load_jsonl_to_dict(path, key='citation'):
    out = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            d = json.loads(line)
            out[d[key]] = d
    return out

print('Loading laws_de classified citations ...')
laws_classified = load_jsonl_to_dict(FILES['laws_classified'])
print(f'  {len(laws_classified):,} entries')

print('Loading court_considerations classified citations ...')
court_classified = load_jsonl_to_dict(FILES['court_classified'])
print(f'  {len(court_classified):,} entries')

classified = {**laws_classified, **court_classified}
print(f'  combined: {len(classified):,}')

Loading laws_de classified citations ...
  197,945 entries
Loading court_considerations classified citations ...
  2,416,056 entries
  combined: 2,592,422


In [8]:
# --- 2b. Citation-to-references graph (outgoing edges per source) ---
print('Loading laws_de_links.json ...')
with open(FILES['laws_links'], 'r', encoding='utf-8') as f:
    laws_links_data = json.load(f)
src2refs_laws = {e['source']: e['references'] for e in laws_links_data['source_to_references']}
print(f'  {len(src2refs_laws):,} source nodes')

print('Loading court_considerations_links.json (large) ...')
with open(FILES['court_links'], 'r', encoding='utf-8') as f:
    court_links_data = json.load(f)
src2refs_court = {e['source']: e['references'] for e in court_links_data['source_to_references']}
print(f'  {len(src2refs_court):,} source nodes')

src2refs = {**src2refs_laws, **src2refs_court}
print(f'  combined: {len(src2refs):,} source nodes')

# free the raw JSON wrappers
del laws_links_data, court_links_data; gc.collect()

Loading laws_de_links.json ...
  31,217 source nodes
Loading court_considerations_links.json (large) ...
  1,125,616 source nodes
  combined: 1,156,833 source nodes


0

## 3. Validate val gold against `gold_citation_coverage.csv`

In [9]:
coverage = pd.read_csv(FILES['coverage'])
val_cov  = coverage[coverage['split'] == 'val']
print(f'Val gold marked retrievable in coverage file: '
      f'{val_cov["present_in_source_column_any"].sum()} / {len(val_cov)}')

val_golds = {}
for _, r in val.iterrows():
    raw = [c.strip() for c in str(r['gold_citations']).split(';')]
    in_corpus = [c for c in raw if c in cit_to_idx]
    val_golds[r['query_id']] = in_corpus
    if len(in_corpus) != len(raw):
        print(f'  {r["query_id"]}: dropped {len(raw)-len(in_corpus)} unretrievable')

print('\nGold counts per val query:')
for qid, gs in val_golds.items():
    print(f'  {qid}: {len(gs)}')

Val gold marked retrievable in coverage file: 222 / 222

Gold counts per val query:
  val_001: 42
  val_002: 36
  val_003: 47
  val_004: 10
  val_005: 11
  val_006: 18
  val_007: 19
  val_008: 29
  val_009: 14
  val_010: 25


## 4. Build enriched corpus passages

Each passage gets four blocks: citation header, structural segments, outgoing references, and the original text. The structure is consistent and machine-friendly so the encoder treats it as a unit.

In [10]:
def format_segments(segments: dict) -> str:
    """Turn the parsed citation segments into a compact human/encoder-readable line."""
    if not segments: return ''
    parts = []
    if 'article' in segments and segments['article']:
        parts.append(f"article {segments['article']}")
    if 'units' in segments and segments['units']:
        # paragraph / subdivision / etc.
        for u in segments['units']:
            if isinstance(u, dict):
                marker = u.get('marker','').strip('.')
                value  = u.get('value','')
                if marker and value:
                    parts.append(f"{marker} {value}")
    if 'law_code' in segments and segments['law_code']:
        parts.append(f"law code {segments['law_code']}")
    if 'volume' in segments and segments['volume']:
        parts.append(f"BGE volume {segments['volume']}")
    if 'series' in segments and segments['series']:
        parts.append(f"series {segments['series']}")
    if 'page' in segments and segments['page']:
        parts.append(f"page {segments['page']}")
    if 'consideration' in segments and segments['consideration']:
        parts.append(f"consideration {segments['consideration']}")
    if 'docket' in segments and segments['docket']:
        parts.append(f"docket {segments['docket']}")
    if 'decision_year' in segments and segments['decision_year']:
        parts.append(f"year {segments['decision_year']}")
    if 'legal_area_code' in segments and segments['legal_area_code']:
        parts.append(f"legal area {segments['legal_area_code']}")
    return ', '.join(parts)

def build_enriched_passage(row, classified, src2refs, max_refs=10, max_text_chars=1600):
    cit   = row['citation']
    title = row['title']
    text  = (row['text'] or '')[:max_text_chars]

    # Header
    if title:
        header = f'[CITATION] {cit} — {title}'
    else:
        header = f'[CITATION] {cit}'

    # Structure block
    cls = classified.get(cit, {})
    family    = cls.get('family', '')
    subfamily = cls.get('subfamily', '')
    pattern   = cls.get('pattern', '')
    seg_str   = format_segments(cls.get('segments', {}))
    structure_bits = [b for b in [family, subfamily, pattern, seg_str] if b]
    structure = '[STRUCTURE] ' + '; '.join(structure_bits) if structure_bits else ''

    # References block (outgoing edges)
    refs = src2refs.get(cit, [])[:max_refs]
    references = '[REFERENCES] ' + '; '.join(refs) if refs else ''

    # Text block
    text_block = f'[TEXT] {text}' if text else ''

    return '\n'.join([b for b in [header, structure, references, text_block] if b])

# Build for the entire corpus
print('Building enriched passages for full corpus ...')
t0 = time.time()
corpus['enriched_text'] = [
    build_enriched_passage(row, classified, src2refs)
    for row in corpus.to_dict(orient='records')
]
print(f'  done in {time.time()-t0:.1f}s')
print(f'  mean enriched length: {int(corpus["enriched_text"].str.len().mean())} chars')

# Sample
print('\n--- SAMPLE enriched passage (laws_de) ---')
print(corpus[corpus['source']=='laws'].iloc[100]['enriched_text'][:800])
print('\n--- SAMPLE enriched passage (court) ---')
print(corpus[corpus['source']=='court'].iloc[100]['enriched_text'][:800])

Building enriched passages for full corpus ...
  done in 18.3s
  mean enriched length: 957 chars

--- SAMPLE enriched passage (laws_de) ---
[CITATION] Art. 38 Abs. 3 131.211 — Verfassung des Kantons Zürich, vom 27. Februar 2005 - Rechtsetzung
[STRUCTURE] law; statute_article; statute_article; article 38, Abs 3, law code 131.211
[TEXT] 3 Verfassung und Gesetz bestimmen, welche Behörden Verordnungen erlassen können.

--- SAMPLE enriched passage (court) ---
[CITATION] BGE 136 I 1 E. 5.4.4
[STRUCTURE] court; bge; court_bge; BGE volume 136, page 1
[REFERENCES] BGE 133 I 249 E. 4.2; BGE 136 I 1 S. 16
[TEXT] Schliesslich ist das Zuchtverbot auch als zumutbar zu beurteilen: Zwar steht auf der einen Seite das private, wirtschaftliche Interesse, Hunde einer gewissen Rasse zu züchten. Auf der anderen Seite ist das gewichtige öffentliche Interesse am Schutz der Allgemeinheit vor gefährlichen Hunden. Angesichts deren bereits dargestellten Gefährlichkeit besteht im vorliegenden Fall ein offensichtli

In [11]:
# Free the auxiliary dicts now that passages are built
del classified, laws_classified, court_classified
del src2refs, src2refs_laws, src2refs_court
gc.collect()

0

## 5. Evaluation framework

In [19]:
def f1(pred_set, gold_set):
    if not pred_set or not gold_set:
        return 0.0
    tp = len(pred_set & gold_set)
    if tp == 0:
        return 0.0
    p = tp / len(pred_set)
    r = tp / len(gold_set)
    return 2 * p * r / (p + r)


def evaluate(query_emb, corpus_emb, val_golds, corpus_cits,
             k_list=(10, 20, 30, 50, 100, 500, 1000),
             batch_q=128):
    import faiss
    import numpy as np
    import pandas as pd

    corpus_emb = np.asarray(corpus_emb, dtype=np.float32)
    query_emb = np.asarray(query_emb, dtype=np.float32)

    assert corpus_emb.ndim == 2
    assert query_emb.ndim == 2
    assert corpus_emb.shape[1] == query_emb.shape[1]
    assert len(corpus_cits) == corpus_emb.shape[0]

    max_k = min(max(k_list), corpus_emb.shape[0])

    print(f"Building CPU FAISS IndexFlatIP for {corpus_emb.shape[0]:,} docs...")
    index = faiss.IndexFlatIP(corpus_emb.shape[1])
    index.add(corpus_emb)

    rows = []
    qids = list(val_golds.keys())

    print(f"Searching {query_emb.shape[0]:,} queries, top-{max_k}...")

    for start in range(0, query_emb.shape[0], batch_q):
        end = min(start + batch_q, query_emb.shape[0])
        D, I = index.search(query_emb[start:end], max_k)

        for local_i, q_i in enumerate(range(start, end)):
            qid = qids[q_i]
            gold_set = set(val_golds[qid])
            if not gold_set:
                continue

            ranked = [corpus_cits[idx] for idx in I[local_i] if idx >= 0]

            row = {
                "qid": qid,
                "n_gold": len(gold_set),
                "F1_oracle_k": f1(set(ranked[:len(gold_set)]), gold_set),
            }

            for fk in (20, 30):
                row[f"F1_k{fk}"] = f1(set(ranked[:fk]), gold_set)

            for k in k_list:
                kk = min(k, len(ranked))
                row[f"R@{k}"] = len(set(ranked[:kk]) & gold_set) / len(gold_set)

            rows.append(row)

    return pd.DataFrame(rows)

def report(df, label):
    print(f'\n=== {label} ===')
    print(df.to_string(index=False, float_format='%.3f'))
    print(f'\n  Macro F1 oracle-k : {df["F1_oracle_k"].mean():.3f}')
    print(f'  Macro F1 k=20     : {df["F1_k20"].mean():.3f}')
    print(f'  Macro F1 k=30     : {df["F1_k30"].mean():.3f}')
    for k in [10,50,100,500,1000]:
        print(f'  Mean Recall@{k:<5}: {df[f"R@{k}"].mean():.3f}')

## 6. Encoding utility (chunked, checkpointed)

In [13]:
def encode_chunked(model, texts, batch_size, chunk_size, save_prefix, prompt_prefix=None,
                   sort_by_length=True):
    """
    Speed-optimized chunked encoder:
      - sort_by_length=True: groups similar-length docs in each batch, minimizing padding.
        Often 2-4x speedup on heterogeneous corpora (statutes mixed with court considerations).
      - Restores original order before saving so npy aligns with corpus row indices.
      - torch.inference_mode() is slightly faster than no_grad.
    """
    os.makedirs(os.path.dirname(save_prefix) or '.', exist_ok=True)
    n = len(texts); n_chunks = math.ceil(n / chunk_size); parts = []
    total_t = 0.0

    for c in range(n_chunks):
        path = f'{save_prefix}_chunk{c:03d}.npy'
        if os.path.exists(path):
            print(f'  [cached] chunk {c+1}/{n_chunks}')
            parts.append(np.load(path)); continue

        s, e = c*chunk_size, min((c+1)*chunk_size, n)
        sub = texts[s:e]
        if prompt_prefix is not None:
            sub = [prompt_prefix + t for t in sub]

        # Length-sorted batching: massive speedup when texts vary in length
        if sort_by_length:
            order = sorted(range(len(sub)), key=lambda i: len(sub[i]))
            sub_sorted = [sub[i] for i in order]
        else:
            order = list(range(len(sub)))
            sub_sorted = sub

        t0 = time.time()
        with torch.inference_mode():
            emb_sorted = model.encode(
                sub_sorted, batch_size=batch_size, show_progress_bar=True,
                normalize_embeddings=True, convert_to_numpy=True,
            ).astype(np.float32)

        # Restore original order
        emb = np.empty_like(emb_sorted)
        for new_i, orig_i in enumerate(order):
            emb[orig_i] = emb_sorted[new_i]

        np.save(path, emb); parts.append(emb)
        dt = time.time()-t0; total_t += dt
        eta = total_t/(c+1)*(n_chunks-c-1)
        rate = len(sub) / dt
        print(f'  chunk {c+1}/{n_chunks}  {len(sub)} docs  in  {dt:.1f}s  '
              f'({rate:.0f} docs/s)  ETA {eta/60:.1f} min')

    return np.concatenate(parts, axis=0)

## 7. Encode full corpus + val queries with `Qwen3-Embedding-8B`

In [14]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'Qwen/Qwen3-Embedding-8B'
TAG        = 'qwen3_8b_enriched'
BATCH      = 256       # was 32 — push it; 102GB VRAM fits this easily with bf16+flash-attn
CHUNK      = 100_000
MAX_SEQ    = 768       # was 1024 — most enriched passages fit in 768; ~1.8x faster than 1024

QWEN_INSTRUCT = (
    'Instruct: Given an English-language legal question or scenario about Swiss federal law, '
    'retrieve the Swiss statute articles or federal court decision considerations that are most '
    'directly relevant to answering it. The passage you retrieve will contain a citation header, '
    'structural metadata, the citations it references, and its German legal text.\nQuery: '
)

# Try flash-attention-2 first (2-3x speedup on H100/A100/Blackwell), fall back to sdpa.
try:
    import flash_attn  # noqa: F401
    ATTN_IMPL = 'flash_attention_2'
except ImportError:
    ATTN_IMPL = 'sdpa'
print(f'Attention implementation: {ATTN_IMPL}')

# bfloat16 is faster and more numerically stable than fp16 on H100/A100/Blackwell.
model = SentenceTransformer(
    MODEL_NAME, device=DEVICE,
    model_kwargs={'torch_dtype': torch.bfloat16, 'attn_implementation': ATTN_IMPL},
    tokenizer_kwargs={'padding_side': 'left'},
)
model.max_seq_length = MAX_SEQ
model.eval()

# Optional: torch.compile gives another 1.3-1.7x. Skipped if it errors.
try:
    model[0].auto_model = torch.compile(
        model[0].auto_model, mode='reduce-overhead', fullgraph=False
    )
    print('torch.compile enabled')
except Exception as ex:
    print(f'torch.compile skipped: {ex}')

print(f'Model loaded. Max seq length: {model.max_seq_length}')
print(f'Embedding dim: {model.get_sentence_embedding_dimension()}')

Attention implementation: sdpa


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

torch.compile enabled
Model loaded. Max seq length: 768
Embedding dim: 4096


In [15]:
import os
import glob
import numpy as np

# Encode full enriched corpus
enriched_texts = corpus['enriched_text'].tolist()
corpus_cits    = corpus['citation'].tolist()

save_prefix = f'{ART_DIR}/{TAG}_doc'

# Heuristic: check if embeddings already exist (either merged or chunked)
existing_files = glob.glob(f"{save_prefix}*.npy")

if existing_files:
    print(f"Found existing embeddings ({len(existing_files)} files). Loading instead of re-encoding...")

    # If you saved a single merged file
    merged_path = f"{save_prefix}.npy"
    if os.path.exists(merged_path):
        doc_emb = np.load(merged_path, mmap_mode="r")
    else:
        # Otherwise load and concatenate chunked files
        parts = sorted(existing_files)
        doc_emb = np.concatenate([np.load(p, mmap_mode="r") for p in parts], axis=0)

else:
    print(f'Encoding {len(enriched_texts):,} enriched passages with {MODEL_NAME} ...')

    doc_emb = encode_chunked(
        model,
        enriched_texts,
        batch_size=BATCH,
        chunk_size=CHUNK,
        save_prefix=save_prefix,
        prompt_prefix=None,  # Qwen3-Embedding does NOT prefix passages
    )

print(f'Doc embeddings shape: {doc_emb.shape}  ({doc_emb.nbytes/1e9:.1f} GB)')

Found existing embeddings (22 files). Loading instead of re-encoding...
Doc embeddings shape: (2161111, 4096)  (35.4 GB)


In [16]:
# Encode val queries
print('Encoding val queries ...')
q_emb = model.encode(
    [QWEN_INSTRUCT + q for q in val['query'].tolist()],
    batch_size=4, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True,
).astype(np.float32)
print(f'Query embeddings shape: {q_emb.shape}')

del model; gc.collect(); torch.cuda.empty_cache()

Encoding val queries ...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Query embeddings shape: (10, 4096)


## 8. Evaluate against full corpus

In [20]:
df = evaluate(q_emb, doc_emb, val_golds, corpus_cits)
report(df, f'{MODEL_NAME} | enriched passages | full corpus ({len(corpus_cits):,} docs)')
df.to_csv(f'{ART_DIR}/results_{TAG}.csv', index=False)

Building CPU FAISS IndexFlatIP for 2,161,111 docs...
Searching 10 queries, top-1000...

=== Qwen/Qwen3-Embedding-8B | enriched passages | full corpus (2,161,111 docs) ===
    qid  n_gold  F1_oracle_k  F1_k20  F1_k30  R@10  R@20  R@30  R@50  R@100  R@500  R@1000
val_001      42        0.000   0.000   0.000 0.000 0.000 0.000 0.000  0.000  0.048   0.167
val_002      36        0.000   0.000   0.000 0.000 0.000 0.000 0.000  0.028  0.028   0.083
val_003      47        0.000   0.000   0.000 0.000 0.000 0.000 0.000  0.000  0.064   0.106
val_004      10        0.000   0.067   0.050 0.000 0.100 0.100 0.300  0.400  0.600   0.600
val_005      11        0.091   0.065   0.049 0.091 0.091 0.091 0.091  0.091  0.364   0.364
val_006      18        0.111   0.105   0.083 0.111 0.111 0.111 0.222  0.222  0.500   0.611
val_007      19        0.105   0.103   0.082 0.053 0.105 0.105 0.105  0.105  0.211   0.263
val_008      29        0.034   0.041   0.034 0.034 0.034 0.034 0.069  0.103  0.103   0.103
val_009   

## 9. Verdict

In [21]:
best_f1   = max(df['F1_oracle_k'].mean(), df['F1_k20'].mean(), df['F1_k30'].mean())
best_r100 = df['R@100'].mean()
best_r500 = df['R@500'].mean()

print('='*92)
print('FINAL VERDICT — Dense embedding maximum-context ceiling on val (n=10), full corpus')
print('='*92)
print(f'  Model              : {MODEL_NAME}')
print(f'  Passages enriched  : citation header + structure + outgoing refs + text')
print(f'  Corpus size        : {len(corpus_cits):,}')
print(f'  Best Macro F1      : {best_f1:.3f}')
print(f'  Mean Recall@100    : {best_r100:.3f}')
print(f'  Mean Recall@500    : {best_r500:.3f}')
print(f'  Target window      : 0.600 – 0.800')
print('-'*92)
if best_f1 >= 0.60:
    print('VERDICT  : Dense embedding REACHES the target — Observation 3 must be revised.')
    print('NEXT STEP: Make this the primary retrieval mechanism. Add reranker only if precision needs lifting.')
elif best_r100 >= 0.70:
    print('VERDICT  : Dense F1 below target, but Recall@100 is high enough to use as a recall stage.')
    print('NEXT STEP: Use dense as candidate generator (k≈100–500); add LLM reranker / cross-encoder.')
else:
    print('VERDICT  : Dense embedding INSUFFICIENT — Observation 3 CONFIRMED with maximum context.')
    print('NEXT STEP: Move to LLM agentic retrieval / structured legal-knowledge pipeline.')
    print('           Even SOTA + every available signal cannot bridge the legal-conceptual gap.')

FINAL VERDICT — Dense embedding maximum-context ceiling on val (n=10), full corpus
  Model              : Qwen/Qwen3-Embedding-8B
  Passages enriched  : citation header + structure + outgoing refs + text
  Corpus size        : 2,161,111
  Best Macro F1      : 0.044
  Mean Recall@100    : 0.117
  Mean Recall@500    : 0.247
  Target window      : 0.600 – 0.800
--------------------------------------------------------------------------------------------
VERDICT  : Dense embedding INSUFFICIENT — Observation 3 CONFIRMED with maximum context.
NEXT STEP: Move to LLM agentic retrieval / structured legal-knowledge pipeline.
           Even SOTA + every available signal cannot bridge the legal-conceptual gap.


## 10. Per-query diagnostics

In [22]:
diag = df.merge(val[['query_id','query']], left_on='qid', right_on='query_id')
diag['query_preview'] = diag['query'].str.slice(0, 140) + '...'
diag = diag[['qid','n_gold','F1_oracle_k','F1_k20','R@50','R@100','R@500','query_preview']]
diag = diag.sort_values('F1_oracle_k', ascending=False)
print(diag.to_string(index=False, float_format='%.3f'))

    qid  n_gold  F1_oracle_k  F1_k20  R@50  R@100  R@500                                                                                                                                   query_preview
val_006      18        0.111   0.105 0.222  0.222  0.500 On 3 March 2012, homeowners Ms. L and her partner Mr. M asked G, an installer they knew socially who works on domestic heating equipment, to...
val_007      19        0.105   0.103 0.105  0.105  0.211 An heirship claims title to a vintage pocket chronometer known as “The Meridian” that belonged to Ms. Barnes, who died in 2010, and contends...
val_005      11        0.091   0.065 0.091  0.091  0.364 A parent, separated from their co-parent since 2008, has not had custody of the two children (primary custody is with the other parent); unt...
val_009      14        0.071   0.059 0.071  0.143  0.357 A divorced custodial parent lives alone with four children born in 1996, 1998, 2001 and 2006. The non-custodial parent is serving a prison 